In [ ]:
%pip install sentence-transformers rank-bm25 numpy scikit-learn --quiet

In [ ]:
import os, sys, json
import numpy as np
sys.path.append('..')
from dotenv import load_dotenv
import anthropic
from utils.helpers import CLAUDE_SONNET, CLAUDE_HAIKU, CLAUDE_OPUS, DEFAULT_MODEL

load_dotenv()
client = anthropic.Anthropic()
MODEL = DEFAULT_MODEL


## 1. Corpus de Documentos

In [ ]:
if "client" not in dir():
    import sys; sys.path.append('..')
    from dotenv import load_dotenv
    import anthropic
    from utils.helpers import DEFAULT_MODEL
    load_dotenv()
    client = anthropic.Anthropic()
    MODEL = DEFAULT_MODEL

# Corpus de ejemplo: Manual de políticas de empresa
DOCUMENTOS = [
    {
        "id": "doc_001",
        "titulo": "Política de Vacaciones",
        "contenido": "Los empleados tienen derecho a 15 días hábiles de vacaciones anuales. Para solicitar vacaciones, se debe enviar la solicitud al menos 2 semanas antes de la fecha deseada. Las vacaciones se acumulan a razón de 1.25 días por mes trabajado. No se pueden tomar más de 10 días consecutivos sin aprobación especial del director."
    },
    {
        "id": "doc_002",
        "titulo": "Política de Trabajo Remoto",
        "contenido": "Los empleados pueden trabajar de forma remota hasta 3 días por semana. Para trabajar desde casa se requiere una conexión a internet estable de al menos 50 Mbps. El empleado debe estar disponible en horario de 9am a 6pm. Se deben asistir presencialmente a las reuniones de equipo los martes."
    },
    {
        "id": "doc_003",
        "titulo": "Beneficios de Salud",
        "contenido": "La empresa cubre el 80% del seguro médico para el empleado y 50% para dependientes directos. Se incluye cobertura dental y visual. Los empleados tienen acceso a un psicólogo de empresa de forma gratuita (hasta 10 sesiones anuales). El seguro aplica desde el primer día de trabajo."
    },
    {
        "id": "doc_004",
        "titulo": "Código de Conducta",
        "contenido": "Todos los empleados deben tratar a sus colegas con respeto y profesionalismo. Está prohibido el acoso en cualquier forma. Los conflictos deben reportarse a Recursos Humanos. El uso de dispositivos personales para trabajo debe hacerse con VPN de empresa. La información confidencial no puede compartirse fuera de la organización."
    },
    {
        "id": "doc_005",
        "titulo": "Plan de Desarrollo Profesional",
        "contenido": "La empresa destina $1,500 anuales por empleado para formación y desarrollo. Las certificaciones técnicas aprobadas son reembolsadas al 100%. Cada empleado tiene una reunión de desarrollo trimestral con su manager. Los empleados pueden dedicar hasta 10% de su tiempo laboral a proyectos de innovación interna."
    },
]

print(f"Corpus cargado: {len(DOCUMENTOS)} documentos")


## 2. Chunking (Fragmentación de Texto)

In [ ]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 50) -> list[str]:
    """Divide texto en chunks con overlap para preservar contexto."""
    words = text.split()
    chunks = []
    start = 0
    
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    
    return chunks

# Crear chunks de todos los documentos
all_chunks = []
for doc in DOCUMENTOS:
    chunks = chunk_text(doc["contenido"], chunk_size=50, overlap=10)
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "chunk_id": f"{doc['id']}_chunk_{i}",
            "doc_id": doc["id"],
            "titulo": doc["titulo"],
            "texto": chunk
        })

print(f"✅ {len(all_chunks)} chunks generados de {len(DOCUMENTOS)} documentos")
print(f"\nEjemplo de chunk:")
print(all_chunks[0])

## 3. BM25 Lexical Search

In [ ]:
from rank_bm25 import BM25Okapi

# Tokenizar documentos para BM25
corpus_tokenizado = [chunk["texto"].lower().split() for chunk in all_chunks]
bm25 = BM25Okapi(corpus_tokenizado)

def buscar_bm25(query: str, top_k: int = 3) -> list[dict]:
    """Busca los chunks más relevantes usando BM25."""
    tokens_query = query.lower().split()
    scores = bm25.get_scores(tokens_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    resultados = []
    for idx in top_indices:
        if scores[idx] > 0:
            resultados.append({
                **all_chunks[idx],
                "score_bm25": float(scores[idx])
            })
    return resultados

# Prueba
query = "¿Cuántos días de vacaciones tengo?"
resultados = buscar_bm25(query, top_k=3)
print(f"Query: '{query}'")
print(f"\nTop {len(resultados)} resultados BM25:")
for r in resultados:
    print(f"  [{r['score_bm25']:.2f}] {r['titulo']}: {r['texto'][:80]}...")

## 4. Embeddings Semánticos

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Cargando modelo de embeddings...")
# Modelo multilingüe pequeño para ejemplos
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("✅ Modelo cargado")

# Crear embeddings para todos los chunks
textos = [chunk["texto"] for chunk in all_chunks]
embeddings = embed_model.encode(textos, show_progress_bar=True)
print(f"✅ Embeddings creados: {embeddings.shape}")

In [ ]:
def buscar_semantico(query: str, top_k: int = 3) -> list[dict]:
    """Busca usando similitud semántica de embeddings."""
    query_embedding = embed_model.encode([query])
    scores = cosine_similarity(query_embedding, embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    return [
        {**all_chunks[idx], "score_semantico": float(scores[idx])}
        for idx in top_indices
    ]

# Prueba con pregunta en diferente formulación
query = "¿Tengo derecho a hacer home office?"
resultados_sem = buscar_semantico(query, top_k=3)
print(f"Query semántica: '{query}'")
for r in resultados_sem:
    print(f"  [{r['score_semantico']:.3f}] {r['titulo']}: {r['texto'][:80]}...")

## 5. Pipeline RAG Completo

In [ ]:
def rag_query(pregunta: str, top_k: int = 3, metodo: str = "semantico") -> str:
    """Pipeline RAG completo: recuperar contexto + generar respuesta."""
    
    # 1. Recuperar documentos relevantes
    if metodo == "semantico":
        chunks_relevantes = buscar_semantico(pregunta, top_k)
    elif metodo == "bm25":
        chunks_relevantes = buscar_bm25(pregunta, top_k)
    else:  # híbrido
        sem = buscar_semantico(pregunta, top_k)
        bm = buscar_bm25(pregunta, top_k)
        # Combinar y deduplicar
        seen = set()
        chunks_relevantes = []
        for c in sem + bm:
            if c['chunk_id'] not in seen:
                seen.add(c['chunk_id'])
                chunks_relevantes.append(c)
    
    # 2. Construir contexto
    contexto = "\n\n".join([
        f"[{c['titulo']}]\n{c['texto']}"
        for c in chunks_relevantes
    ])
    
    # 3. Generar respuesta con Claude
    system = """
    Eres un asistente de Recursos Humanos. Responde las preguntas de empleados 
    basándote ÚNICAMENTE en los documentos de política de empresa proporcionados.
    Si la información no está en los documentos, dilo claramente.
    """
    
    prompt = f"""
<documentos_de_politica>
{contexto}
</documentos_de_politica>

<pregunta_empleado>
{pregunta}
</pregunta_empleado>

Responde de forma clara y concisa, citando la política relevante.
"""
    
    response = client.messages.create(
        model=MODEL, max_tokens=500,
        system=system,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.content[0].text

# Prueba del pipeline RAG
preguntas = [
    "¿Cuántos días de vacaciones me corresponden?",
    "¿Puedo trabajar desde casa todos los días?",
    "¿La empresa paga mi seguro médico?",
    "¿Cuánto me dan para tomar cursos?",
]

for pregunta in preguntas:
    print(f"\n{'='*60}")
    print(f"👤 Empleado: {pregunta}")
    print(f"🤖 RR.HH:   {rag_query(pregunta)}")